# PingTracer export: a closer look

This notebook reads a CSV saved from PingTracer (**Export > Ping data to CSV**) and walks through it:

1. the numbers for each host
2. a histogram of ping times for each host
3. the latency spikes, and which hosts spiked together
4. one spike up close

It is a front end only. The work is done by the scripts beside it, which also run on their own from a terminal:
`ping_export.py` (reading and statistics), `ping_report.py`, `ping_histogram.py`, `ping_spikes.py`, and `test_ping_tools.py` to check them.

**To use it:** change `EXPORT` in the next cell to your file, then run every cell from the top.

In [ ]:
from pathlib import Path
import sys

sys.path.insert(0, str(Path.cwd()))   # the scripts sit in the same folder as this notebook

EXPORT = Path("../dist/analysis-demo/real-export.csv")   # <-- your export goes here
MIN_OUTAGE = 2        # lost pings in a row that count as an outage
SPIKE_ABOVE_MS = None  # how far above the median is a spike; None = 10 ms or 3x the usual spread, whichever is larger

## 1. Read the export

In [ ]:
import ping_export
import ping_report
import ping_histogram
import ping_spikes

hosts = ping_export.read_export(EXPORT)
summaries = {host: ping_export.summarise(host, pings, MIN_OUTAGE) for host, pings in hosts.items()}

for host, pings in hosts.items():
    print(f"{len(pings):7d} pings   {host}")

## 2. The numbers

All times are in milliseconds.

- **baseline**: the fastest ping. The path with nothing queued.
- **worst 1%**: 99 pings in 100 were this fast or faster.
- **jitter**: the average change from one ping to the next.
- **queueing**: the average time above the baseline.

In [ ]:
ms = ping_export.ms
columns = ("pings", "lost %", "baseline", "median", "mean", "95%", "worst 1%", "max", "jitter", "queueing")
print(f"{'host':<34}" + "".join(f"{c:>10}" for c in columns))
for host, s in summaries.items():
    row = (s.pings, f"{s.loss_percent:.2f}", ms(s.baseline), ms(s.median), ms(s.mean), ms(s.p95), ms(s.p99), ms(s.maximum), ms(s.jitter), ms(s.queueing_mean))
    print(f"{host[:33]:<34}" + "".join(f"{str(v):>10}" for v in row))

The same, in full, with every drop-out and any reply whose data came back altered:

In [ ]:
for summary in summaries.values():
    print(ping_report.report(summary))
    print()

## 3. Histograms

One picture per host. The top panel counts how many pings took each time, with the baseline, median and worst 1% marked.
The bottom panel is the same pings in the order they happened, with lost pings in red.

If a few pings are far slower than the rest they are left off the histogram, with a note saying how many and how slow, so the main shape stays readable. They still appear in the bottom panel.

In [ ]:
import matplotlib.pyplot as plt

for host, pings in hosts.items():
    figure = ping_histogram.figure_host(summaries[host], pings, ping_export.choose_times(pings))
    if figure is None:
        print(f"{host}: no replies, nothing to draw")
    else:
        plt.show()

## 4. Spikes

A spike is a ping well above the host's own median. Spike pings less than a second apart count as one spike.

For each spike this lists the hosts that spiked at the same moment. That is a pointer for where to look: a spike that also shows on the first host began at or before it, and one that only the later hosts saw began further out. It comes from timing alone and is not proof of a cause.

In [ ]:
spikes = ping_spikes.analyse(hosts, above_ms=SPIKE_ABOVE_MS)
print(ping_spikes.report(spikes))

Every host on one time axis. Orange diamonds are spike pings, the dashed line is each host's spike threshold, and a shaded band marks a moment when more than one host spiked.

In [ ]:
figure = ping_spikes.figure_spikes(hosts, spikes)
if figure is None:
    print("no replies, nothing to draw")
else:
    plt.show()

## 5. One spike up close

Takes the highest spike in the file and lists every host's pings for a few seconds either side of it, so the order things happened in can be read off directly. Change `WINDOW_SECONDS`, or set `moment` yourself to look at a different time.

In [ ]:
from datetime import timedelta

WINDOW_SECONDS = 2.0

all_spikes = [spike for host_spikes in spikes for spike in host_spikes.spikes]
if not all_spikes:
    print("no spikes in this file")
else:
    biggest = max(all_spikes, key=lambda spike: spike.peak_ms - next(h.median_ms for h in spikes if h.host == spike.host))
    moment = biggest.start
    print(f"Highest spike: {biggest.host}, {ms(biggest.peak_ms)} ms at {moment:%H:%M:%S.%f}"[:-3])
    print(f"Pings from {WINDOW_SECONDS:g} s before it to {WINDOW_SECONDS:g} s after its end (* = spike, lost = no reply):")
    print()
    thresholds = {h.host: h.threshold_ms for h in spikes}
    for host, pings in hosts.items():
        print(f"  {host}   (median {ms(summaries[host].median)} ms)")
        for ping in pings:
            if moment - timedelta(seconds=WINDOW_SECONDS) <= ping.sent <= biggest.end + timedelta(seconds=WINDOW_SECONDS):
                offset = (ping.sent - moment).total_seconds()
                if not ping.replied:
                    value, mark = "lost", ""
                else:
                    value, mark = f"{ms(ping.rtt_ms)} ms", " *" if ping.rtt_ms > thresholds[host] else ""
                print(f"    {offset:+7.2f} s  {value:>8}{mark}")
        print()

## Checking the scripts

`test_ping_tools.py` builds an export with known contents and compares every figure with a value worked out a second way, and with NumPy where it is installed. Run it from a terminal in this folder:

```
py -3.13 test_ping_tools.py
```